# Stereo-Seq - spc022 100M

## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import gc
import spatialdata_plot
from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

matplotlib.use("Agg")

import logging
logging.getLogger('ome_zarr').setLevel(logging.ERROR) # This mutes unnecessary Zarr warning (see: https://github.com/scverse/spatialdata/issues/400)

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _stereoseq import stereoseq

## Create sdata

In [ ]:
dataset_id = "spc022"
path_SAW = f"/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/{dataset_id}/subsampled_100M/SAW/{dataset_id}"
SN = "Y40048N5"

regist_img_path = os.path.join(path_SAW, f"outs/image/{SN}_ssDNA_regist.tif")
raw_h5ad_path = os.path.join(path_SAW, f"outs/feature_expression/{SN}.raw.h5ad")
tissue_cut_path = os.path.join(path_SAW, f"outs/image/{SN}_ssDNA_tissue_cut.tif")
tissue_h5ad_path = [
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin1.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin5.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin10.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin20.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin50.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin100.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin150.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin200.h5ad"),
]
output_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/{dataset_id}/subsampled_100M/harpy/sdata.zarr'

barcodeToPos_path = f"/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/raw/Stereo-Seq/{dataset_id}/mask/{SN}.barcodeToPos.h5"

pipeline_name = 'SAW'
species = 'mouse'
tissue_annotation_file = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/{dataset_id}/tissue_annotation/tissue.geojson'
capture_area_file = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/{dataset_id}/tissue_annotation/capture_area.geojson'

In [5]:
# sdata = sd.read_zarr(output_path)
# sdata

In [6]:
sdata = stereoseq(
    regist_img_path = regist_img_path,
    dataset_id = dataset_id,
    raw_h5ad_path = raw_h5ad_path,
    tissue_h5ad_path = tissue_h5ad_path,
    tissue_cut_path = tissue_cut_path,
    output = output_path,
)
sdata

2026-02-04 09:36:37.088 | WARNING  | harpy.io._spatial_data:_load_image_to_dask:231 - dims parameter is equal to ['c', 'y', 'x'], but will be ignored when reading in images from a file
2026-02-04 09:36:46.262 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'spc022_image'
2026-02-04 09:36:52.902 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'spc022_tissue_cut'
/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/spatialdata/models/models.py:1183: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(adata)


INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/spatialdata/models/models.py:1183: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(adata)


INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/spatialdata/models/models.py:1183: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(adata)


...

INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      
INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/spatialdata/models/models.py:1183: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(adata)


INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/spatialdata/models/models.py:1183: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(adata)


SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/spc022/harpy/sdata.zarr
├── Images
│     └── 'spc022_image': DataTree[cyx] (1, 23520, 23520), (1, 11760, 11760), (1, 5880, 5880), (1, 2940, 2940), (1, 1470, 1470)
├── Labels
│     ├── 'spc022_bin_1_labels': DataArray[yx] (19363, 11898)
│     ├── 'spc022_bin_5_labels': DataArray[yx] (3874, 2381)
│     ├── 'spc022_bin_10_labels': DataArray[yx] (1938, 1191)
│     ├── 'spc022_bin_20_labels': DataArray[yx] (969, 596)
│     ├── 'spc022_bin_50_labels': DataArray[yx] (388, 239)
│     ├── 'spc022_bin_100_labels': DataArray[yx] (194, 120)
│     ├── 'spc022_bin_150_labels': DataArray[yx] (130, 81)
│     ├── 'spc022_bin_200_labels': DataArray[yx] (98, 61)
│     ├── 'spc022_raw_labels': DataArray[yx] (23520, 23520)
│     └── 'spc022_tissue_cut': DataTree[yx] (23520, 23520), (11760, 11760), (5880, 5880), (2940, 2940), (1470, 1470)
├── Shapes
│     ├── 

In [6]:
levels = [level.split(f'{dataset_id}_')[1].split("_table")[0] for level in [*sdata.tables]]
print(f"{pipeline_name} bins: {levels}")

SAW bins: ['bin_100', 'bin_10', 'bin_150', 'bin_1', 'bin_200', 'bin_20', 'bin_50', 'bin_5', 'raw']


In [7]:
# Assert whether number of shapes matches number of label IDs
import dask.array as da

for level in levels:
    number_of_shapes = len(sdata.shapes[f"{dataset_id}_{level}_shapes"])
    number_of_label_IDs = len(da.unique(sdata.labels[f"{dataset_id}_{level}_labels"].data).compute()) - 1
    assert number_of_shapes == number_of_label_IDs, f"{level}: Number of shapes ({number_of_shapes}) does not match number of label IDs ({number_of_label_IDs})"

## Calculate pixel size

In [8]:
microns_per_pixel = 0.500
print(f"Pixel size: {microns_per_pixel:.3f} µm")

Pixel size: 0.500 µm


## Hexagonal tesselation

In [9]:
custom_bins = ['hexagonal_grid_20um', 'hexagonal_grid_120um'] # Names refer to the diagonal of the hexagon

In [ ]:
shape = (sdata.images[f"{dataset_id}_image"]['scale0']['image'].shape[1], sdata.images[f"{dataset_id}_image"]['scale0']['image'].shape[2])

for custom_bin in custom_bins:
    gc.collect()
    
    size = (int(custom_bin.split("hexagonal_grid_")[1].split("um")[0])/2) / microns_per_pixel # Radius of the hexagon in pixels
    
    # Create hexagonal grids
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape,
        size=size,
        output_shapes_layer=f"{custom_bin}_shapes", 
        output_labels_layer=f"{custom_bin}_labels", 
        grid_type="hexagon", 
        offset=(0, 0), 
        chunks=2048, 
        client=None, 
        transformations=get_transformation(sdata.shapes[f"{dataset_id}_raw_shapes"], get_all=True),
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

    # Rebin data based on hexagonal grid
    sdata = hp.tb.bin_counts( 
        sdata,
        table_layer=f"{dataset_id}_raw_table",
        labels_layer=f"{custom_bin}_labels",
        output_layer=f"{custom_bin}_table",
        append = False,
        to_coordinate_system = dataset_id,
        overwrite=True,
    )

2026-02-04 14:07:04.573 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'hexagonal_grid_20um_labels'
2026-02-04 14:07:46.877 | INFO     | harpy.table._allocation:_aggregate:488 - Calculating cell counts.
2026-02-04 14:14:35.442 | INFO     | harpy.table._allocation:bin_counts:363 - 
 Number of spots per bin  Frequency
                       1      20110
                       2       2271
                       3        360
                       4        174
                       5        155
                       6        132
                       7        134
                       8        108
                       9        126
                      10        113
                      11        106
                      12        141
                      13        100
                      14        105
                      15         96
                      16         98
                      17        105
                      18         93
       

## Specify plotting parameters

In [ ]:
DPI = 300
MAX_DIM_FOR_PLOTS = 10000

HE_image = f"{dataset_id}_image"

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

## Create crop for zoomed-in plots

In [ ]:
min_x = sdata.images[HE_image]['scale0']['image'].shape[2] / 2 # Center of image
max_x = min_x + round(200 / microns_per_pixel) # 200 micrometer
min_y = sdata.images[HE_image]['scale0']['image'].shape[1] / 2 # Center of image
max_y = min_y + round(200 / microns_per_pixel) # 200 micrometer

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y], 
    max_coordinate=[max_x, max_y], 
    axes=("x", "y"), 
    target_coordinate_system=dataset_id
)

figsize_crop = (3,3)

## Plot ssDNA

In [ ]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "ssDNA"

sdata.pl.render_images(
    element=HE_image, scale=best_scale, cmap='gray'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'ssDNA.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

## Plot SAW tissue segmentation mask

In [ ]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = f"{pipeline_name} - Tissue segmentation mask"

sdata.pl.render_images(
    element=HE_image, scale=best_scale, cmap='gray'
).pl.render_labels(
    element=f"{dataset_id}_tissue_cut", fill_alpha=0.3, contour_px=rel_fontsize(fig, 0.003), outline_alpha=1, scale=best_scale
).pl.show(
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_segmentation_mask.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

## Plot SAW bins

In [ ]:
for level in levels:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - {level}"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale, cmap='gray'
    ).pl.render_shapes(
            element=f'{dataset_id}_{level}_shapes', fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"{pipeline_name} - {level}"
    
    sdata_crop.pl.render_images(
        element=HE_image, scale="full", cmap='gray' 
    ).pl.render_shapes(
            element=f'{dataset_id}_{level}_shapes', fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize_crop,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

# NOTE: Spots are not represented true to size. For the smallest level, the spots are represented as squares with a side length of 500 nm while the DNBs are actually smaller with a spot size of 220 nm diameter. 

## Plot SAW QC metrics

In [ ]:
# Plot QC metrics
for level in levels:
    for qc_metric in ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']:
        gc.collect()
        
        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{dataset_id}_{level}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"{pipeline_name} - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale, cmap='gray'
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=False,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )
        
        sdata.pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{dataset_id}_{level}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'{pipeline_name}_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

## Plot hexagonal spots

In [ ]:
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale, cmap='gray'
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata_crop.pl.render_images(
        element=HE_image, scale="full", cmap='gray' 
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Spots are represented true to scale


## Create bounding box for total capture area

In [ ]:
# Add bounding box for total capture area
gdf = gpd.read_file(capture_area_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.images[f"{dataset_id}_image"], get_all=True),
    overwrite = True,
)

In [ ]:
# Plot capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale, cmap='gray'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.05)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Plot capture area bbox
for level in ["raw"]:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} {level} - Capture area bounding box"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale, cmap='gray'
    ).pl.render_shapes(
        element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
    ).pl.render_shapes(
        element=f'{dataset_id}_{level}_shapes', fill_alpha=0, outline_width=1, outline_alpha=1, outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'capture_area_bbox.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

In [ ]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

In [ ]:
from math import dist
import shapely

geoms = sdata.shapes[f"{dataset_id}_capture_area"].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

## Add tissue annotation

In [ ]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.images[f"{dataset_id}_image"], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

In [ ]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale, cmap='gray'
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'Qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{dataset_id}_{level}_table",
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{custom_bin}_table", 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

In [ ]:
# Check spots in tissue
for level in levels:
    gc.collect()

    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{level} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale, cmap='gray'
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=f"{dataset_id}_{level}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

In [ ]:
# Check spots in tissue
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{custom_bin} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale, cmap='gray'
    ).pl.render_labels(
        element=f"{custom_bin}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=f"{custom_bin}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

## QC

In [ ]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{dataset_id}_{level}_table",
        species = species
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{custom_bin}_table",
        species = species
    )

In [ ]:
# Plot QC metrics
for level in levels:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{dataset_id}_{level}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale, cmap='gray'
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=False,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )
        
        sdata.pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{dataset_id}_{level}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

In [ ]:
# Plot QC metrics
for custom_bin in custom_bins:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, custom_bin)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{custom_bin}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale, cmap='gray'
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=False,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        sdata.pl.render_labels(
            element=f"{custom_bin}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{custom_bin}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

## Leiden clustering

In [ ]:
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY

for custom_bin in custom_bins:
    gc.collect()
    
    print(f'running: {custom_bin}', flush=True)

    labels_layer=f"{custom_bin}_labels"
    shapes_layer=f"{custom_bin}_shapes"
    table_layer=f"{custom_bin}_table"
    
    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)
    
    # Only keep spots in tissue for clustering
    adata = sdata.tables[table_layer][sdata.tables[table_layer].obs['in_tissue_annotation'] == True]
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=f"{table_layer}_processed",
        region=adata.obs[_REGION_KEY].cat.categories.to_list(),
        overwrite=True,
    )
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False,
        labels_layer=labels_layer,
        table_layer=f"{table_layer}_processed",
        output_layer=f"{table_layer}_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps=50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.75, 1, 1.25]
    calculate_umap = True
    calculate_neighbors = True

    for res in leiden_resolutions:
        gc.collect()

        res_str = str(res).replace('.', '')
        print(f'Leiden resolution: {res}', flush=True)
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_processed",
            output_layer=f"{table_layer}_processed",
            calculate_umap=calculate_umap,
            calculate_neighbors=calculate_neighbors,
            rank_genes=False,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        calculate_umap = False # Only calculate UMAP once
        calculate_neighbors = False # Only calculate neighbors once

        # Plot Leiden clustering
        gc.collect()
        
        with matplotlib.rc_context({'legend.markerscale': 2}):
            fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

            title = f"Leiden {res}"

            sdata.pl.render_images(
                element=HE_image, scale=best_scale, cmap='gray'
            ).pl.render_labels(
                element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, cmap="rainbow", color=f'leiden_{res_str}', table_name=f"{table_layer}_processed", scale=best_scale
            ).pl.show(
                coordinate_systems=dataset_id,
                title=title,
                figsize=figsize,
                colorbar=False,
                legend_fontsize=rel_fontsize(fig, 0.015), 
                legend_fontweight='bold', 
                legend_loc='right margin',
                na_in_legend=False,
                dpi=DPI,
                ax=ax,
            )

            ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
            ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
            ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
            ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

            save_path = os.path.join(level_plots_path, f'Leiden_{res}.png')
            fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

            plt.close(fig)

            print("Plot saved at:", save_path, flush=True)
        
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(level_plots_path, f'Leiden_{res}')
        )

## Spots metrics

In [ ]:
gc.collect()

In [ ]:
import h5py

tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
mask_shape = tissue_mask.shape

barcode_rows = []
chunk_size = 2048

with h5py.File(barcodeToPos_path, "r") as f:
    dset = f["bpMatrix_1"]
    n_y, n_x, _ = dset.shape

    for y_start in range(0, n_y, chunk_size):
        y_end = min(y_start + chunk_size, n_y)
        slab = dset[y_start:y_end, :, 0] 
        ys, xs = np.nonzero(slab)
        ys = ys + y_start

        df_chunk = pd.DataFrame({
            "x": xs,
            "y": ys,
        })

        df_chunk["barcode"] = df_chunk["x"].astype(str) + "_" + df_chunk["y"].astype(str)

        df_chunk["in_tissue_annotation"] = df_chunk.apply(
            lambda row: assign_region((row["x"], row["y"]), tissue_mask, tissue_mask.shape), axis=1
        )

        barcode_rows.append(df_chunk)

barcode_df = pd.concat(barcode_rows, ignore_index=True)

# Total number of spots in capture area
print(f"Shape of barcode whitelist: ({n_y}, {n_x})")
print(f"Total number of possible coordinates: {n_y * n_x:,}") # NOTE: This includes track lines
print(f"Total number of spots in capture area: {len(barcode_df):,}")

# Total number of spots in tissue
total_n_spots_in_tissue = len(barcode_df[barcode_df['in_tissue_annotation'] == 1])
print(f"Total number of spots in tissue: {total_n_spots_in_tissue:,}")

In [ ]:
# Total number of spots in table
n_spots = len(sdata.tables[f"{dataset_id}_raw_table"].obs)
print(f"Number of spots in table: {n_spots:,}")

In [ ]:
# Number of spots in tissue (in table)
n_spots_in_tissue = len(sdata.tables[f"{dataset_id}_raw_table"][sdata.tables[f"{dataset_id}_raw_table"].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue:,}")

In [ ]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in barcode_df['barcode']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in barcode_df[barcode_df['in_tissue_annotation'] == 1]['barcode']:
        f.write(f"{item}\n")      

In [ ]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

## Capture effeciency metrics

In [ ]:
gc.collect()

In [ ]:
def row_sums(X):
    """Return 1D array of row sums without densifying."""
    return np.asarray(X.sum(axis=1)).ravel()

def col_sums(X):
    """Return 1D array of column sums without densifying."""
    return np.asarray(X.sum(axis=0)).ravel()

In [ ]:
# Define Anndata
adata = sdata.tables[f"{dataset_id}_raw_table"]

In [ ]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = (row_sums(adata.X) > 0).sum()
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero:,}")

In [ ]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = adata.obs['in_tissue_annotation'] == 1
umis_mask = row_sums(adata.X) > 0
n_spots_in_tissue_not_zero = (in_tissue_mask & umis_mask).sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero:,}")

In [ ]:
# Total number of genes in table
print(f"Total number of genes in table: {(col_sums(adata.X) > 0).sum():,}")

In [ ]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(col_sums(adata[adata.obs['in_tissue_annotation'] == 1].X) > 0).sum():,}")

In [ ]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum():,.0f}")

In [ ]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum():,.0f}")

In [ ]:
raw_barcode_gene_exp = pd.read_csv(os.path.join(path_SAW, f'outs/feature_expression/{SN}_raw_barcode_gene_exp.txt'), delimiter='\t')
# NOTE: An annotated list file with the information of coordinate, gene, MID, read counts, which is prepared to be a sampling file that performs sequence saturation.

print(f"Total number of UMIs: {raw_barcode_gene_exp['readCount'].sum():,}")
unique_UMIs = raw_barcode_gene_exp[['x', 'y', 'geneID', 'MIDIndex']].drop_duplicates().shape[0]
print(f"Total number of unique UMIs: {unique_UMIs:,}")